# 02 - XLM-RoBERTa Fine-Tuning for Wildlife Trade Detection
**Project:** Illegal Wildlife Trade (IWT) Detection System (SDG 15)  
**Model:** `facebook/xlm-roberta-base`  
**Dataset:** Mou et al. WLT-OSN (ICWSM 2024) `share_data.csv`  

### Overview
This notebook trains an XLM-RoBERTa sequence classification model with:
1. Group-aware train/validation splitting (`GroupShuffleSplit` on `source_group` to prevent label leakage).
2. Class-weighted Cross-Entropy Loss to handle the ~3% positive class imbalance.
3. Memory optimization (gradient checkpointing, small batch size, and frozen embeddings for 4GB VRAM constraint).
4. Benchmarking against the TF-IDF baseline (`data/processed/baseline_metrics.json`).
5. Exporting model weights to `backend/models/checkpoints/xlm-roberta-iwt/`.

In [ ]:
import sys
import os
import json
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# Locate repository root
REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT / "backend"))

from data.loader import load_wlt_osn, group_aware_split
from models.text_classifier import IWTDataset, compute_eval_metrics, get_training_args

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active training device: {DEVICE}")

## 1. Load Data and Group-Aware Split

In [ ]:
csv_path = REPO_ROOT / "data" / "raw" / "share_data.csv"
df = load_wlt_osn(csv_path)

# Split by source_group with seed=0
train_df, test_df = group_aware_split(df, test_size=0.2, seed=0)

print(f"Train size: {len(train_df)} | Positives: {train_df['label'].sum()}")
print(f"Test size:  {len(test_df)}  | Positives: {test_df['label'].sum()}")

# Compute class weights for loss function
n_neg = (train_df['label'] == 0).sum()
n_pos = (train_df['label'] == 1).sum()
pos_weight = n_neg / n_pos
print(f"Positive class weight: {pos_weight:.2f}")

## 2. Tokenization
Tokenize with `xlm-roberta-base` at `max_length=128`.

In [ ]:
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_df['text'].tolist(), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_df['text'].tolist(), truncation=True, padding=True, max_length=128)

train_dataset = IWTDataset(train_encodings, train_df['label'].tolist())
test_dataset = IWTDataset(test_encodings, test_df['label'].tolist())

## 3. Model Initialization & Weighted Loss Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# If on limited local GPU (e.g. 4GB GTX 1650), freeze embeddings to prevent OOM
if DEVICE == "cuda" and torch.cuda.get_device_properties(0).total_memory < 5e9:
    print("Low VRAM detected: Freezing embedding layer")
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

# Custom Trainer with Weighted Cross-Entropy Loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = torch.tensor([1.0, float(pos_weight)], device=model.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 4. Fine-Tuning Execution

In [ ]:
output_dir = REPO_ROOT / "backend" / "models" / "checkpoints" / "xlm-roberta-iwt"
training_args = get_training_args(str(output_dir), num_epochs=3, batch_size=8)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_eval_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Run training (best performed on Kaggle/Colab GPU)
# trainer.train()
print("Training configuration ready for Kaggle / Colab GPU execution.")

## 5. Qualitative Prediction Verification

In [ ]:
sample_sentences = [
    "pangolin scales for sale, dm for price, discreet shipping",
    "spotted a pangolin at Kanha tiger reserve, beautiful creature",
    "carved ivory tusk antique piece available for wholesale",
    "working with WWF on anti-poaching camera traps in Assam"
]

for s in sample_sentences:
    inputs = tokenizer(s, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        prob = torch.softmax(outputs.logits, dim=-1)[0, 1].item()
        print(f"Text: {s[:50]}... -> Suspected Prob: {prob:.4f}")